# 02 EDA

Exploratory analysis for offer-response modeling. This notebook focuses on distribution checks, target behavior, and the main report-ready business patterns.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data/processed'
FIGURES_DIR = PROJECT_ROOT / 'reports/figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
sns.set_palette(sns.color_palette(['#00704A', '#27251F', '#D4E9E2', '#CBA258', '#F2F0EB']))
sns.set_theme(style='whitegrid')


In [ ]:
response = pd.read_parquet(DATA_DIR / 'response_merged.parquet')
features = pd.read_parquet(DATA_DIR / 'features.parquet')

join_keys = ['person', 'offer_id', 'received_time', 'label']
eda = response.merge(
    features[
        join_keys + [
            'age_imputed',
            'income_imputed',
            'membership_duration_days',
            'n_transactions_before',
            'avg_spend_before',
            'offer_completion_rate_before',
        ]
    ],
    on=join_keys,
    how='left',
    validate='one_to_one',
)

eda['gender_clean'] = eda['gender'].fillna('Unknown')
eda['income_decile'] = pd.qcut(eda['income_imputed'], 10, labels=False, duplicates='drop') + 1
eda['membership_quantile'] = pd.qcut(eda['membership_duration_days'], 4, duplicates='drop')
membership_labels = ['Newest', 'Mid-New', 'Mid-Old', 'Oldest'][: len(eda['membership_quantile'].cat.categories)]
eda['membership_quantile'] = eda['membership_quantile'].cat.rename_categories(membership_labels)
eda['transaction_frequency_band'] = pd.qcut(eda['n_transactions_before'], 4, duplicates='drop')
transaction_labels = ['Low', 'Mid-Low', 'Mid-High', 'High'][: len(eda['transaction_frequency_band'].cat.categories)]
eda['transaction_frequency_band'] = eda['transaction_frequency_band'].cat.rename_categories(transaction_labels)

eda[['person', 'offer_type', 'gender_clean', 'income_decile', 'membership_quantile']].head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].hist(eda['age_imputed'], bins=20, color='#00704A', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')

axes[1].hist(eda['income_imputed'], bins=20, color='#27251F', edgecolor='white')
axes[1].set_title('Income Distribution')
axes[1].set_xlabel('Income')

axes[2].hist(eda['membership_duration_days'], bins=20, color='#CBA258', edgecolor='white')
axes[2].set_title('Membership Duration Distribution')
axes[2].set_xlabel('Membership Duration Days')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_demographic_distributions.png', bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

target_rate = eda['label'].value_counts(normalize=True).sort_index()
axes[0].bar(['No Complete', 'Complete'], target_rate.values, color=['#D4E9E2', '#00704A'])
axes[0].set_title('Target Distribution')
axes[0].set_ylabel('Share')

gender_rates = eda.groupby('gender_clean')['label'].mean().sort_values(ascending=False)
axes[1].bar(gender_rates.index, gender_rates.values, color='#27251F')
axes[1].set_title('Completion Rate by Gender')
axes[1].set_ylabel('Completion Rate')

offer_rates = eda.groupby('offer_type')['label'].mean().sort_values(ascending=False)
axes[2].bar(offer_rates.index, offer_rates.values, color='#00704A')
axes[2].set_title('Completion Rate by Offer Type')
axes[2].set_ylabel('Completion Rate')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_target_gender_offer.png', bbox_inches='tight')
plt.show()


In [ ]:
channel_rates = pd.Series({
    'web': eda.loc[eda['channel_web'] == 1, 'label'].mean(),
    'email': eda.loc[eda['channel_email'] == 1, 'label'].mean(),
    'mobile': eda.loc[eda['channel_mobile'] == 1, 'label'].mean(),
    'social': eda.loc[eda['channel_social'] == 1, 'label'].mean(),
})

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(channel_rates.index, channel_rates.values, color='#00704A')
ax.set_title('Completion Rate by Channel Exposure')
ax.set_ylabel('Completion Rate')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_channel_completion.png', bbox_inches='tight')
plt.show()


In [ ]:
income_rates = eda.groupby('income_decile', as_index=False)['label'].mean()
membership_rates = eda.groupby('membership_quantile', as_index=False)['label'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(income_rates['income_decile'], income_rates['label'], marker='o', color='#00704A')
axes[0].set_title('Completion Rate by Income Decile')
axes[0].set_xlabel('Income Decile')
axes[0].set_ylabel('Completion Rate')

axes[1].bar(membership_rates['membership_quantile'], membership_rates['label'], color='#27251F')
axes[1].set_title('Completion Rate by Membership Tenure')
axes[1].set_ylabel('Completion Rate')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_income_membership_completion.png', bbox_inches='tight')
plt.show()


In [ ]:
offer_design = (
    eda.groupby(['offer_type', 'difficulty', 'reward'], as_index=False)['label']
    .mean()
    .rename(columns={'label': 'completion_rate'})
)
transaction_band = (
    eda.groupby('transaction_frequency_band', as_index=False)['label']
    .mean()
    .dropna()
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
scatter = axes[0].scatter(
    offer_design['difficulty'],
    offer_design['reward'],
    c=offer_design['completion_rate'],
    s=200,
    cmap='YlGn',
    edgecolor='black',
)
axes[0].set_title('Offer Design vs Completion Rate')
axes[0].set_xlabel('Difficulty')
axes[0].set_ylabel('Reward')
fig.colorbar(scatter, ax=axes[0], label='Completion Rate')

axes[1].bar(transaction_band['transaction_frequency_band'], transaction_band['label'], color='#00704A')
axes[1].set_title('Completion Rate by Transaction Frequency')
axes[1].set_ylabel('Completion Rate')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_offer_design_transaction_frequency.png', bbox_inches='tight')
plt.show()


In [ ]:
corr_columns = [
    'label',
    'age_imputed',
    'income_imputed',
    'membership_duration_days',
    'n_transactions_before',
    'avg_spend_before',
    'offer_completion_rate_before',
    'difficulty',
    'reward',
]
corr = eda[corr_columns].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(corr, cmap='YlGnBu', center=0, annot=True, fmt='.2f', ax=ax)
ax.set_title('Correlation Heatmap for Core Modeling Features')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_correlation_heatmap.png', bbox_inches='tight')
plt.show()

eda.groupby('offer_type')['label'].agg(['mean', 'count']).sort_values('mean', ascending=False)
